# 04 – Cells and ligand–receptor links for run A (stroma → early lymphoid)

**Question.** Which cells and which L–R links go into the first CausalCCC run?

Decisions taken before this notebook:
- merged groups: senders = all kept stromal cells, receivers = all kept early lymphoid cells (notebook 02);
- at most **500 cells per population**, so that cTEC and DN do not dominate;
- L–R links: the paper's Notch claims are always included; the remaining links are ranked by the **S5 mean**
  (the authors' CellPhoneDB value, read from the correctly oriented column, notebook 03);
- two clean-up rules while filling the list: a pair is skipped if all its gene links are already in the list
  (several S5 rows describe the same genes), and pairs whose "receptor" is IDE (insulin-degrading enzyme, an enzyme
  rather than a signalling receptor) are excluded;
- metadata variables: population, donor and chemistry, plus sequencing depth (`n_counts`, total counts per cell).
  Depth was first left out (it is partly a consequence of expression, the sum of all counts), but a check of the first
  run showed a module of housekeeping genes linked through depth, including neighbours of NOTCH1 (notebook 05), so it
  is included as a variable.

This notebook only prepares the inputs. The CausalCCC steps (gene selection, mosaic table, MIIC) run in R in notebook 05,
as in case study 1. R has no h5ad reader installed, so the counts are exported in Matrix Market format.

In [1]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import anndata as ad
import scipy.io

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
RES = ROOT / "results/thymus_early"
OUT = RES / "04_runA_input"
OUT.mkdir(parents=True, exist_ok=True)

CAP = 500
SEED = 0
N_LINKS = 25                      # L-R pairs (S5 rows); CausalCCC networks usually have about 20-30 links
METADATA = ["population", "donor", "method", "n_counts"]
MAX_GOI = 30                      # causalCCC.MIselection(): at most 30 genes of interest per side

## 1. Cap the cells at 500 per population

Cells are drawn at random (fixed seed) within each population, so the donor mix of each population is kept on average.

In [2]:
cells = pd.read_csv(RES / "02_cell_populations/selected_cells.csv", index_col=0)
capped = pd.concat([d.sample(n=min(len(d), CAP), random_state=SEED) for _, d in cells.groupby("population")])
capped["group"] = capped.lineage.map({"stroma": "stroma", "early lymphoid": "early_lymphoid"})

summary = pd.DataFrame({"before": cells.population.value_counts(), "after": capped.population.value_counts()})
summary["lineage"] = cells.groupby("population").lineage.first()
summary.sort_values(["lineage", "after"], ascending=[False, False])

,before,after,lineage
population,,,
Fb_1,822,500,stroma
cTEC,9030,500,stroma
mTEC(II),322,322,stroma
Endo,233,233,stroma
DN(P),5700,500,early lymphoid
DN(Q),5329,500,early lymphoid
NK,995,500,early lymphoid
γδT,1518,500,early lymphoid
ILC3,294,294,early lymphoid


In [3]:
print(capped.group.value_counts().to_string())
# Donor share per group, before and after the cap
pd.concat({"before": pd.crosstab(cells.lineage, cells.donor, normalize="index"),
           "after": pd.crosstab(capped.lineage, capped.donor, normalize="index")}).round(2)

group
early_lymphoid    2622
stroma            1555


donor                   C34   C40   C41   F22   F23   F64   F74
       lineage                                                 
before early lymphoid  0.00  0.44  0.22  0.05  0.11  0.10  0.08
       stroma          0.01  0.56  0.34  0.01  0.04  0.02  0.02
after  early lymphoid  0.01  0.54  0.17  0.05  0.09  0.07  0.08
       stroma          0.02  0.44  0.16  0.05  0.15  0.10  0.07

**Reading.** After the cap, the stromal group has 1,555 cells: cTEC and Fb_1 at 500 each, mTEC(II) and Endo with
all their cells. cTEC drops from 87% to 32% of the group. The lymphoid group has 2,622 cells. Within the stroma, C40
and C41 still provide most cells, because cTEC and mTEC(II) come almost only from them.

## 2. Choose the L–R links

CausalCCC links **genes**. A CellPhoneDB complex (e.g. `IL7 receptor` = IL7R + IL2RG, `CD94:NKG2A` = KLRD1 + KLRC1)
is replaced by its subunit genes, and the ligand is linked to each subunit. The list is then filled in order
(claimed pairs first, then by S5 mean), applying the two clean-up rules.

In [4]:
DB = THY / "cellphonedb_v2_db/db_2019-09"
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")

def genes_of(name):
    if name in cplx.index:
        units = cplx.loc[name, ["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()
        return [gene.gene_name[u] for u in units]
    return [name]                                    # already a gene symbol (notebook 03)

cand = pd.read_csv(RES / "03_ligand_receptor_candidates/candidates_A_stroma_to_lymphoid.csv")
CLAIMED = ["JAG1_NOTCH1", "JAG2_NOTCH1", "DLL1_NOTCH1", "DLL4_NOTCH1"]
EXCLUDED_RECEPTORS = {"IDE"}
assert set(CLAIMED) <= set(cand.pair)

cand["claimed"] = cand.pair.isin(CLAIMED)
ranked = cand.sort_values(["claimed", "s5_max_mean"], ascending=[False, False]).reset_index(drop=True)

chosen, skipped, gene_links = [], [], set()
for _, row in ranked.iterrows():
    if len(chosen) == N_LINKS:
        break
    new_links = {(l, r) for l in genes_of(row.ligand) for r in genes_of(row.receptor)}
    if set(genes_of(row.receptor)) & EXCLUDED_RECEPTORS:
        skipped.append((row.pair, "receptor is an enzyme (IDE)"))
    elif new_links <= gene_links:
        skipped.append((row.pair, "all gene links already included"))
    else:
        chosen.append(row.pair)
        gene_links |= new_links

links = ranked[ranked.pair.isin(chosen)].copy()
cols = ["pair", "ligand", "receptor", "orientation_rule", "claimed", "s5_max_mean", "s5_best_pair",
        "max_ligand_frac", "ligand_pops", "max_receptor_frac", "receptor_pops"]
print("skipped while filling the list:", skipped)
links[cols].round(2)

skipped while filling the list: [('IGF2_IDE', 'receptor is an enzyme (IDE)'), ('CD94:NKG2C_HLAE_HUMAN', 'all gene links already included')]


,pair,ligand,receptor,orientation_rule,claimed,s5_max_mean,s5_best_pair,max_ligand_frac,ligand_pops,max_receptor_frac,receptor_pops
0,DLL4_NOTCH1,DLL4,NOTCH1,receptor flag,True,0.38,Endo -> DN(early),0.49,"cTEC, Endo",0.58,"ETP, DN(early), DN(P), DN(Q), NK, γδT"
1,JAG1_NOTCH1,JAG1,NOTCH1,receptor flag,True,0.32,Endo -> DN(early),0.28,"cTEC, mTEC(II), Endo",0.58,"ETP, DN(early), DN(P), DN(Q), NK, γδT"
2,JAG2_NOTCH1,JAG2,NOTCH1,receptor flag,True,0.29,cTEC -> DN(early),0.56,"cTEC, mTEC(II), Endo",0.58,"ETP, DN(early), DN(P), DN(Q), NK, γδT"
3,DLL1_NOTCH1,DLL1,NOTCH1,receptor flag,True,0.26,Endo -> DN(early),0.25,Endo,0.58,"ETP, DN(early), DN(P), DN(Q), NK, γδT"
4,CD74_MIF,MIF,CD74,receptor flag,False,2.38,cTEC -> ETP,0.94,"cTEC, mTEC(II), Fb_1, Endo",0.74,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT"
5,CCL19_CCR7,CCL19,CCR7,receptor flag,False,1.99,mTEC(II) -> γδT,0.93,"cTEC, mTEC(II)",0.18,"ETP, NK, γδT"
6,APP_CD74,APP,CD74,receptor flag,False,1.99,Endo -> ETP,0.89,"cTEC, mTEC(II), Fb_1, Endo",0.74,"ETP, DN(early), DN(P), DN(Q), ILC3, NK, γδT"
7,CCL19_CXCR3,CCL19,CXCR3,receptor flag,False,1.96,mTEC(II) -> NK,0.93,"cTEC, mTEC(II)",0.24,NK
8,IGF2_IGF2R,IGF2,IGF2R,receptor flag,False,1.86,Fb_1 -> NK,1.00,"mTEC(II), Fb_1, Endo",0.27,"DN(early), DN(P), DN(Q), NK, γδT"
9,IGF1R_IGF2,IGF2,IGF1R,receptor flag,False,1.86,Fb_1 -> DN(P),1.00,"mTEC(II), Fb_1, Endo",0.41,"ETP, DN(early), DN(P), DN(Q), ILC3, γδT"


In [5]:
# Where the cut falls: the next candidates that were not included
last = ranked.index[ranked.pair == chosen[-1]][0]
ranked.iloc[last + 1:last + 7][["pair", "orientation_rule", "s5_max_mean", "s5_best_pair"]].round(2)

,pair,orientation_rule,s5_max_mean,s5_best_pair
27,SELL_PODXL,ambiguous,0.96,Endo -> DN(early)
28,IL15 receptor_IL15,receptor flag,0.95,mTEC(II) -> NK
29,JAM2_a4b1 complex,integrin,0.93,Endo -> γδT
30,SELE_CD44,receptor flag,0.93,Endo -> ILC3
31,a4b1 complex_SPP1,integrin,0.89,Endo -> γδT
32,HBEGF_CD44,manual,0.87,Endo -> ILC3


In [6]:
edges = pd.DataFrame([dict(ligands=l, receptors=r, pair=row.pair)
                      for _, row in links.iterrows()
                      for l in genes_of(row.ligand) for r in genes_of(row.receptor)]).drop_duplicates(["ligands", "receptors"])
ligand_genes, receptor_genes = sorted(edges.ligands.unique()), sorted(edges.receptors.unique())
print(len(links), "L-R pairs ->", len(edges), "gene-gene links")
print("ligand genes:  ", len(ligand_genes), ligand_genes)
print("receptor genes:", len(receptor_genes), receptor_genes)
print("genes of interest per side:", len(ligand_genes), "(senders),", len(receptor_genes), "(receivers); limit", MAX_GOI)
assert len(ligand_genes) <= MAX_GOI and len(receptor_genes) <= MAX_GOI

25 L-R pairs -> 27 gene-gene links
ligand genes:   18 ['APP', 'CCL19', 'CCL25', 'CD34', 'CLEC2D', 'CXCL12', 'DLK1', 'DLL1', 'DLL4', 'FN1', 'HLA-E', 'IGF2', 'IL7', 'JAG1', 'JAG2', 'MIF', 'PTN', 'SPP1']
receptor genes: 22 ['CCR7', 'CCR9', 'CD44', 'CD74', 'CXCR3', 'CXCR4', 'IGF1R', 'IGF2R', 'IL2RG', 'IL7R', 'ITGA4', 'ITGB1', 'KLRB1', 'KLRC1', 'KLRC2', 'KLRD1', 'NOTCH1', 'NOTCH2', 'NOTCH3', 'PLXNB2', 'SELL', 'TNFRSF14']
genes of interest per side: 18 (senders), 22 (receivers); limit 30


In [7]:
# Genes that are both a ligand and a receptor in this set (CausalCCC suffixes them _senders / _receivers)
sorted(set(ligand_genes) & set(receptor_genes))

[]

## 3. Export for R (notebook 05)

- `counts.mtx.gz`: raw counts, genes × cells (all 33,694 genes, because the gene selection in notebook 05 ranks every
  gene). It is rebuilt by this notebook and not tracked by Git (`*.mtx.gz` in `.gitignore`).
- `genes.csv`, `cells.csv`: row and column names; `cells.csv` also holds the group (sender / receiver), population,
  donor, chemistry and total counts per cell (`n_counts`, computed by the authors on the raw counts).
- `interact_edges.tsv`: the gene–gene L–R links (columns `ligands`, `receptors`, as `causalCCC.wrapper()` expects).
- `selected_links.csv`: the 25 S5 pairs with their ranking information.

34 gene symbols occur twice in the matrix (notebook 03). R requires unique names, so the second copy gets a `-1`
suffix (`var_names_make_unique`). None of them is a ligand or receptor.

In [8]:
raw = ad.read_h5ad(THY / "zenodo_5500511/HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")
rows = np.sort(raw.obs_names.get_indexer(capped.index))
assert (rows >= 0).all()
sub = raw[rows].to_memory()
sub.var_names_make_unique()
assert set(ligand_genes + receptor_genes) <= set(sub.var_names)

with gzip.open(OUT / "counts.mtx.gz", "wb") as f:
    scipy.io.mmwrite(f, sub.X.T.tocoo().astype(np.int32), field="integer")
pd.Series(sub.var_names, name="gene").to_csv(OUT / "genes.csv", index=False)
capped.loc[sub.obs_names, ["group"] + METADATA + ["Age", "lineage"]].rename_axis("cell").to_csv(OUT / "cells.csv")
edges[["ligands", "receptors"]].to_csv(OUT / "interact_edges.tsv", sep="\t", index=False)
links[cols].to_csv(OUT / "selected_links.csv", index=False)
print(sub.shape, "->", sorted(p.name for p in OUT.iterdir()))

/tmp/ipykernel_433064/641972841.py:1: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "zenodo_5500511/HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


/tmp/ipykernel_433064/641972841.py:4: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  sub = raw[rows].to_memory()


(4177, 33694) -> ['cells.csv', 'counts.mtx.gz', 'genes.csv', 'interact_edges.tsv', 'selected_links.csv']


## Summary

| | |
|---|---|
| Senders (`stroma`) | 1,555 cells: cTEC 500 · Fb_1 500 · mTEC(II) 322 · Endo 233 |
| Receivers (`early_lymphoid`) | 2,622 cells: DN(P), DN(Q), NK, γδT 500 each · ILC3 294 · DN(early) 201 · ETP 127 |
| L–R pairs | 25 S5 pairs = 4 claimed Notch pairs + 21 by S5 mean → 27 gene–gene links |
| Skipped while filling | `IGF2_IDE` (IDE is an enzyme), `CD94:NKG2C_HLAE_HUMAN` (its gene links were already in) |
| Genes of interest | 18 ligands, 22 receptors – within the limit of 30 |
| Metadata variables | population, donor, chemistry, sequencing depth (`n_counts`) |
| Exported to | `results/thymus_early/04_runA_input/` |

**Points to keep in mind when reading the network:**
- Ranking by the S5 mean favours genes that are expressed almost everywhere (MIF, CD74, APP, HLA-E, IGF2). This was
  checked: ranking by expression fraction instead does not change that, and penalises the most specific claims more
  (see the discussion before this notebook). The claimed pairs are therefore forced in, and the rest is ranked by S5.
- The stromal group remains dominated by C40/C41 cells (60%), as expected from notebook 02.
- Sequencing depth (`n_counts`) is a variable in the network, to absorb depth-driven links between genes.